# LLM 급식 메뉴 추천 RAG 프로젝트

## 1. 데이터 로드

급식 메뉴, 메뉴별 식재료, 식품 영양성분 데이터를 결합하여
LLM/RAG 검색에 사용할 메뉴 데이터셋을 구축한다.

In [2]:
import re
import numpy as np
import pandas as pd

from pathlib import Path

In [48]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [3]:
# 현재 프로젝트 위치
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"

print("현재 작업 폴더 :", BASE_DIR)
print("dataset 경로  :", DATA_DIR)
print("dataset 존재  :", DATA_DIR.exists())

print("\n=== dataset 파일 목록 ===")

for file in DATA_DIR.iterdir():
    print(f"{file.name:40} {file.stat().st_size:,} bytes")

현재 작업 폴더 : d:\script\LLM-PRJ
dataset 경로  : d:\script\LLM-PRJ\dataset
dataset 존재  : True

=== dataset 파일 목록 ===
food_nutrition.xlsx                      13,348,408 bytes
menugen_menu_ingredients.csv             3,361,795 bytes
menugen_menu_master.csv                  218,450 bytes


## 2. 메뉴 마스터 데이터 확인

MenuGen 메뉴 마스터 데이터를 불러오고
메뉴 수, 컬럼 구조, 결측치 여부를 확인한다.

In [4]:
menu_path = DATA_DIR / "menugen_menu_master.csv"

menu_df = pd.read_csv(menu_path)

print("menu_df shape :", menu_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(menu_df.columns):
    print(i, col)

print("\n=== 상위 5행 ===")
display(menu_df.head())

print("\n=== 결측치 개수 ===")
display(menu_df.isnull().sum())

menu_df shape : (3250, 7)

=== 컬럼 목록 ===
0 no
1 fd_Code
2 upper_Fd_Grupp_Nm
3 fd_Grupp_Nm
4 fd_Nm
5 fd_Wgh
6 food_Cnt

=== 상위 5행 ===


,no,fd_Code,upper_Fd_Grupp_Nm,fd_Grupp_Nm,fd_Nm,fd_Wgh,food_Cnt
0,1,D011001,밥류,쌀밥,눌은밥,440.0,1
1,2,D011002,밥류,쌀밥,쌀밥,210.0,1
2,3,D011003,밥류,쌀밥,찰밥,210.0,1
3,4,D011004,밥류,쌀밥,현미밥,210.0,1
4,5,D011006,밥류,쌀밥,현미밥,160.0,1



=== 결측치 개수 ===


no                   0
fd_Code              0
upper_Fd_Grupp_Nm    0
fd_Grupp_Nm          0
fd_Nm                0
fd_Wgh               0
food_Cnt             0
dtype: int64

## 3. 메뉴별 식재료 데이터 확인

메뉴 코드(`fd_Code`)를 기준으로 각 메뉴에 연결된 식재료 데이터를 불러오고,
행 수, 컬럼 구조, 결측치를 확인한다.

In [5]:
ingredient_path = DATA_DIR / "menugen_menu_ingredients.csv"

ingredient_df = pd.read_csv(ingredient_path)

print("ingredient_df shape :", ingredient_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(ingredient_df.columns):
    print(i, col)

print("\n=== 상위 10행 ===")
display(ingredient_df.head(10))

print("\n=== 결측치 개수 ===")
display(ingredient_df.isnull().sum())

print("\n=== 고유 메뉴 코드 수 ===")
print(ingredient_df["menu_fd_Code"].nunique())

ingredient_df shape : (22582, 15)

=== 컬럼 목록 ===
0 menu_fd_Code
1 menu_fd_Nm
2 menu_upper_Fd_Group_Nm
3 menu_fd_Group_Nm
4 menu_fd_Wgh
5 ingredient_fd_Code
6 ingredient_food_Code
7 ingredient_food_Nm
8 ingredient_fd_Eng_Nm
9 ingredient_nation_Std_Food_Grupp_Code_Nm
10 ingredient_origin_Code_Nm
11 ingredient_food_Wgh
12 ingredient_allrgy_Info
13 ingredient_onslf_Std_Food_Grupp_Nm
14 ingredient_amplt_Cl_Nm

=== 상위 10행 ===


,menu_fd_Code,menu_fd_Nm,menu_upper_Fd_Group_Nm,menu_fd_Group_Nm,menu_fd_Wgh,ingredient_fd_Code,ingredient_food_Code,ingredient_food_Nm,ingredient_fd_Eng_Nm,ingredient_nation_Std_Food_Grupp_Code_Nm,ingredient_origin_Code_Nm,ingredient_food_Wgh,ingredient_allrgy_Info,ingredient_onslf_Std_Food_Grupp_Nm,ingredient_amplt_Cl_Nm
0,D011001,눌은밥,NaN,NaN,440.0,D011001,F00093,"즉석밥, 누룽지, 끓는물 부음","Instant cooked rice, Scorched rice(Nurungji), ...",곡류 및 그 제품,농진청,440.0,NaN,NaN,NaN
1,D011002,쌀밥,NaN,NaN,210.0,D011002,F00079,"멥쌀, 백미, 밥","Rice(Ssal), White, Cooked",곡류 및 그 제품,농진청,210.0,NaN,NaN,NaN
2,D011003,찰밥,NaN,NaN,210.0,D011003,F03123,"찹쌀, 백미, 보람찰, 밥","Rice(Ssal), Glutinous, White, Boramchal, Cooked",곡류 및 그 제품,농진청,210.0,NaN,NaN,NaN
3,D011004,현미밥,NaN,NaN,210.0,D011004,F00081,"멥쌀, 현미, 밥","Rice(Ssal), Brown, Cooked",곡류 및 그 제품,농진청,210.0,NaN,NaN,NaN
4,D011006,현미밥,NaN,NaN,160.0,D011006,F00081,"멥쌀, 현미, 밥","Rice(Ssal), Brown, Cooked",곡류 및 그 제품,농진청,160.0,NaN,NaN,NaN
5,D011007,누룽지(멥쌀),NaN,NaN,100.0,D011007,F00014,메밀묵,"Memilmuk, Buckwheat starch jelly",곡류 및 그 제품,농진청,100.0,메밀,NaN,NaN
6,D011008,흑미밥,NaN,NaN,90.0,D011008,F00017,"멥쌀, 백미, 생것","Rice(Ssal), White, Raw",곡류 및 그 제품,대푯값,70.0,NaN,NaN,NaN
7,D011008,흑미밥,NaN,NaN,90.0,D011008,F03154,"멥쌀, 현미, 흑미, 생것","Rice(Ssal), Black, Raw",곡류 및 그 제품,농진청,20.0,NaN,NaN,NaN
8,D011009,발아현미밥,NaN,NaN,100.0,D011009,F00017,"멥쌀, 백미, 생것","Rice(Ssal), White, Raw",곡류 및 그 제품,대푯값,90.0,NaN,NaN,NaN
9,D011009,발아현미밥,NaN,NaN,100.0,D011009,F00025,"멥쌀, 현미, 발아현미, 생것","Rice(Ssal), Brown, Germinated, Raw",곡류 및 그 제품,농진청,10.0,NaN,NaN,NaN



=== 결측치 개수 ===


menu_fd_Code                                    0
menu_fd_Nm                                      0
menu_upper_Fd_Group_Nm                      22582
menu_fd_Group_Nm                            22582
menu_fd_Wgh                                     0
ingredient_fd_Code                              0
ingredient_food_Code                            0
ingredient_food_Nm                              0
ingredient_fd_Eng_Nm                            0
ingredient_nation_Std_Food_Grupp_Code_Nm        0
ingredient_origin_Code_Nm                    1429
ingredient_food_Wgh                             0
ingredient_allrgy_Info                      17036
ingredient_onslf_Std_Food_Grupp_Nm          22582
ingredient_amplt_Cl_Nm                      22582
dtype: int64


=== 고유 메뉴 코드 수 ===
3250


## 4. 메뉴 마스터와 식재료 데이터 연결 검증

메뉴 코드 기준으로 두 데이터셋이 정상적으로 대응되는지 확인하고,
식재료가 없는 메뉴 또는 메뉴 마스터에 존재하지 않는 코드가 있는지 검증한다.

In [6]:
# 메뉴 마스터 코드
menu_codes = set(menu_df["fd_Code"])

# 식재료 데이터의 메뉴 코드
ingredient_menu_codes = set(ingredient_df["menu_fd_Code"])

print("메뉴 마스터 고유 코드 수 :", len(menu_codes))
print("식재료 데이터 고유 코드 수 :", len(ingredient_menu_codes))

# 메뉴 마스터에는 있지만 식재료 데이터에는 없는 메뉴
missing_ingredient_codes = menu_codes - ingredient_menu_codes

# 식재료 데이터에는 있지만 메뉴 마스터에는 없는 메뉴
unknown_menu_codes = ingredient_menu_codes - menu_codes

print("\n=== 식재료 데이터가 없는 메뉴 ===")
print("개수 :", len(missing_ingredient_codes))
print(list(missing_ingredient_codes)[:20])

print("\n=== 메뉴 마스터에 존재하지 않는 식재료 메뉴 코드 ===")
print("개수 :", len(unknown_menu_codes))
print(list(unknown_menu_codes)[:20])

# 메뉴별 식재료 개수 확인
ingredient_count = (
    ingredient_df
    .groupby("menu_fd_Code")
    .size()
    .sort_values(ascending=False)
)

print("\n=== 메뉴별 식재료 개수 통계 ===")
display(ingredient_count.describe())

print("\n=== 식재료가 가장 많은 메뉴 Top 10 ===")
display(ingredient_count.head(10))

메뉴 마스터 고유 코드 수 : 3250
식재료 데이터 고유 코드 수 : 3250

=== 식재료 데이터가 없는 메뉴 ===
개수 : 0
[]

=== 메뉴 마스터에 존재하지 않는 식재료 메뉴 코드 ===
개수 : 0
[]

=== 메뉴별 식재료 개수 통계 ===


count    3250.000000
mean        6.948308
std         4.603561
min         1.000000
25%         2.000000
50%         7.000000
75%        10.000000
max        27.000000
dtype: float64


=== 식재료가 가장 많은 메뉴 Top 10 ===


menu_fd_Code
D054013    27
D064024    25
D082061    24
D064030    22
D053126    22
D031066    21
D105020    21
D112032    21
D016021    21
D014037    20
dtype: int64

## 5. 국가표준식품성분 데이터 구조 확인

영양성분 엑셀 파일의 시트 목록과 각 시트의 크기를 확인하여
실제 데이터가 존재하는 시트를 식별한다.

In [7]:
nutrition_path = DATA_DIR / "food_nutrition.xlsx"

# 엑셀 파일의 시트 목록 확인
xls = pd.ExcelFile(nutrition_path)

print("=== 시트 목록 ===")
print(xls.sheet_names)

print("\n=== 시트별 데이터 크기 ===")

for sheet in xls.sheet_names:
    temp_df = pd.read_excel(
        nutrition_path,
        sheet_name=sheet
    )
    
    print(f"{sheet} : {temp_df.shape}")

=== 시트 목록 ===
['DB 설명', '국가표준식품성분 Database 10.0', '국가표준식품성분 Database 10.1', '국가표준식품성분 Database 10.2', '국가표준식품성분 Database 10.3', '국가표준식품성분 Database 10.4', 'DB 10.4 신규,교체,삭제 식품목록', '부록1)식품코드 연계표', '부록2)식품코드,국문명,영문명,학명 정보 ', '부록3)영양성분표기및단위']

=== 시트별 데이터 크기 ===
DB 설명 : (0, 0)
국가표준식품성분 Database 10.0 : (3272, 137)
국가표준식품성분 Database 10.1 : (3261, 137)
국가표준식품성분 Database 10.2 : (3312, 137)
국가표준식품성분 Database 10.3 : (3332, 137)
국가표준식품성분 Database 10.4 : (3368, 137)
DB 10.4 신규,교체,삭제 식품목록 : (188, 3)
부록1)식품코드 연계표 : (3369, 15)
부록2)식품코드,국문명,영문명,학명 정보  : (3366, 5)
부록3)영양성분표기및단위 : (135, 5)


## 6. 국가표준식품성분 Database 10.4 로드

국가표준식품성분 데이터 중 최신 버전인 10.4를 기준 데이터로 사용한다.
이후 MenuGen 식재료 데이터와 연결하기 위해 식품코드, 식품명 및 주요 영양성분 컬럼을 확인한다.

In [8]:
nutrition_df = pd.read_excel(
    nutrition_path,
    sheet_name="국가표준식품성분 Database 10.4"
)

print("nutrition_df shape :", nutrition_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(nutrition_df.columns):
    print(i, col)

print("\n=== 상위 5행 ===")
display(nutrition_df.head())

print("\n=== 결측치 개수 상위 30개 ===")
display(
    nutrition_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

nutrition_df shape : (3368, 137)

=== 컬럼 목록 ===
0 Unnamed: 0
1 *음영표시: 책자 수록 항목
2 Unnamed: 2
3 가식부 100g 당 (per 100g Edible Portion)
4 Unnamed: 4
5 일반성분 Proximates
6 Unnamed: 6
7 Unnamed: 7
8 Unnamed: 8
9 Unnamed: 9
10 Unnamed: 10
11 Unnamed: 11
12 Unnamed: 12
13 Unnamed: 13
14 Unnamed: 14
15 Unnamed: 15
16 Unnamed: 16
17 Unnamed: 17
18 Unnamed: 18
19 Unnamed: 19
20 Unnamed: 20
21 무기질 Minerals
22 Unnamed: 22
23 Unnamed: 23
24 Unnamed: 24
25 Unnamed: 25
26 Unnamed: 26
27 Unnamed: 27
28 Unnamed: 28
29 Unnamed: 29
30 Unnamed: 30
31 Unnamed: 31
32 Unnamed: 32
33 비타민 Vitamins
34 Unnamed: 34
35 Unnamed: 35
36 Unnamed: 36
37 Unnamed: 37
38 Unnamed: 38
39 Unnamed: 39
40 Unnamed: 40
41 Unnamed: 41
42 Unnamed: 42
43 Unnamed: 43
44 Unnamed: 44
45 Unnamed: 45
46 Unnamed: 46
47 Unnamed: 47
48 Unnamed: 48
49 Unnamed: 49
50 Unnamed: 50
51 Unnamed: 51
52 Unnamed: 52
53 Unnamed: 53
54 Unnamed: 54
55 Unnamed: 55
56 Unnamed: 56
57 Unnamed: 57
58 Unnamed: 58
59 Unnamed: 59
60 Unnamed: 60
61 Unnamed: 61
62 U

,Unnamed: 0,*음영표시: 책자 수록 항목,Unnamed: 2,가식부 100g 당 (per 100g Edible Portion),Unnamed: 4,일반성분 Proximates,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,식염상당량,폐기율
0,DB10.4\n색인,10개정 \n책자\n색인,식품군,식품명,출처,에너지,수분,단백질,지방,회분,...,도코사\n펜타에노산\n(22:5(n-3)),도코사\n헥사에노산\n(22:6(n-3)),오메가3 \n지방산,오메가6 \n지방산,총 트랜스\n지방산,트랜스 \n올레산(18:1(n-9)t),트랜스 \n리놀레산(18:2t),트랜스 \n리놀렌산(18:3t),NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,kcal,g,g,g,g,...,mg,mg,g,g,g,mg,mg,mg,g,%
2,1,1,곡류 및 그 제품,"귀리, 겉귀리, 도정, 생것",농진청('21),388,11.7,9.88,8.84,1.56,...,0,0,0.08,2.77,0.02,2.75,10.91,4.5,0,0
3,3381,NaN,곡류 및 그 제품,"귀리, 겉귀리, 도정, 밥",농진청('21),197,55.5,5.01,4.7,0.72,...,0,0,0.04,1.4,0.01,1.84,9.06,3.35,0,0
4,2,2,곡류 및 그 제품,"귀리, 쌀귀리, 도정, 생것",농진청('20),388,11.6,11.14,8.9,1.7,...,0,0,0.07,2.8,0.01,2.97,3.58,5.25,0,0



=== 결측치 개수 상위 30개 ===


*음영표시: 책자 수록 항목                         2152
Unnamed: 0                                 1
Unnamed: 2                                 1
가식부 100g 당 (per 100g Edible Portion)       1
Unnamed: 4                                 1
폐기율                                        1
콜레스테롤                                      1
식염상당량                                      1
일반성분 Proximates                            0
Unnamed: 8                                 0
Unnamed: 7                                 0
Unnamed: 6                                 0
Unnamed: 9                                 0
Unnamed: 13                                0
Unnamed: 10                                0
Unnamed: 11                                0
Unnamed: 12                                0
Unnamed: 17                                0
Unnamed: 18                                0
Unnamed: 19                                0
Unnamed: 20                                0
무기질 Minerals                               0
Unnamed: 1

## 7. 국가표준식품성분 DB 10.4 헤더 정제

엑셀의 실제 컬럼명은 두 번째 행에 존재하며,
그 아래 단위 행을 제거하여 실제 식품 데이터만 구성한다.

In [9]:
# DB10.4 원본 로드
nutrition_raw = pd.read_excel(
    nutrition_path,
    sheet_name="국가표준식품성분 Database 10.4",
    header=None
)

# 헤더 구성 정보
upper_header = nutrition_raw.iloc[0]
detail_header = nutrition_raw.iloc[1]
unit_row = nutrition_raw.iloc[2].copy()

# 세부 컬럼명이 있으면 사용하고,
# 비어 있으면 상위분류명을 컬럼명으로 사용
column_names = detail_header.copy()

column_names = column_names.where(
    column_names.notna(),
    upper_header
)


# 컬럼명 정규화 함수
def clean_column_names(columns):
    return (
        pd.Index(columns)
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


column_names = clean_column_names(column_names)

# 실제 식품 데이터
nutrition_df = (
    nutrition_raw
    .iloc[3:]
    .reset_index(drop=True)
)

nutrition_df.columns = column_names

# 단위 정보의 인덱스도 동일한 컬럼명으로 연결
unit_row.index = column_names


print("nutrition_df shape :", nutrition_df.shape)
display(nutrition_df.head(2))

nutrition_df shape : (3366, 137)


1,DB10.4 색인,10개정 책자 색인,식품군,식품명,출처,에너지,수분,단백질,지방,회분,...,도코사 펜타에노산 (22:5(n-3)),도코사 헥사에노산 (22:6(n-3)),오메가3 지방산,오메가6 지방산,총 트랜스 지방산,트랜스 올레산(18:1(n-9)t),트랜스 리놀레산(18:2t),트랜스 리놀렌산(18:3t),식염상당량,폐기율
0,1,1,곡류 및 그 제품,"귀리, 겉귀리, 도정, 생것",농진청('21),388,11.7,9.88,8.84,1.56,...,0,0,0.08,2.77,0.02,2.75,10.91,4.5,0,0
1,3381,NaN,곡류 및 그 제품,"귀리, 겉귀리, 도정, 밥",농진청('21),197,55.5,5.01,4.7,0.72,...,0,0,0.04,1.4,0.01,1.84,9.06,3.35,0,0


## 8. DB10.4 영양성분 표기 규칙

국가표준식품성분 DB10.4 설명문의 표기 기준을 적용한다.

- `-` : 측정되지 않은(unmeasured) 결측값
- `Tr` : 검출되었으나 정량 가능한 최소 농도 이하의 미량(trace)
- `(수치)` : 인용되었거나 재료량을 이용해 환산한 수치
- `(Tr)` : 괄호 표기가 적용된 미량값

원본값은 보존하고 계산용 수치와 데이터 상태를 별도로 관리한다.

In [10]:
def parse_nutrition_value(value):
    if pd.isna(value):
        return np.nan, "missing"

    value = str(value).strip()

    if value == "-":
        return np.nan, "unmeasured"

    if value.lower() == "tr":
        return np.nan, "trace"

    if value.lower() == "(tr)":
        return np.nan, "quoted_or_converted_trace"

    if re.fullmatch(r"\([\d,]+(?:\.\d+)?\)", value):
        numeric_value = value[1:-1].replace(",", "")
        return float(numeric_value), "quoted_or_converted"

    numeric_value = pd.to_numeric(
        value.replace(",", ""),
        errors="coerce"
    )

    if pd.notna(numeric_value):
        return float(numeric_value), "numeric"

    return np.nan, "unknown"

## 9. 영양성분 컬럼 그룹 구성

DB10.4의 상위 헤더 정보를 이용하여 전체 컬럼을 영양성분 그룹별로 분류한다.

전체 컬럼을 개별적으로 확인하지 않고,
영양성분 그룹 단위로 활용 범위를 먼저 선정한 뒤
필요한 세부 컬럼만 추출한다.

In [11]:
# DB10.4 상위 헤더를 각 컬럼에 전달
group_names = nutrition_raw.iloc[0].ffill()

# 상위 헤더명 정규화
group_names = clean_column_names(group_names)

# 컬럼 메타데이터 구성
nutrition_meta_df = pd.DataFrame({
    "column": nutrition_df.columns,
    "group": group_names,
    "unit": unit_row.values
})

# 식품 기본정보는 별도 그룹으로 지정
info_cols = [
    "DB10.4 색인",
    "10개정 책자 색인",
    "식품군",
    "식품명",
    "출처"
]

nutrition_meta_df.loc[
    nutrition_meta_df["column"].isin(info_cols),
    "group"
] = "식품정보"


# 단위가 있는 컬럼 = 수치형 전처리 대상
# 영양성분뿐 아니라 식염상당량, 폐기율도 포함
value_cols = (
    nutrition_meta_df
    .loc[nutrition_meta_df["unit"].notna(), "column"]
    .tolist()
)


print("식품 정보 컬럼 수 :", len(info_cols))
print("수치형 전처리 대상 :", len(value_cols))


# 그룹별 구조 확인
group_summary_df = (
    nutrition_meta_df
    .groupby("group", dropna=False)
    .agg(
        컬럼수=("column", "count"),
        컬럼목록=("column", list)
    )
    .reset_index()
)

display(group_summary_df)

식품 정보 컬럼 수 : 5
수치형 전처리 대상 : 132


,group,컬럼수,컬럼목록
0,무기질 Minerals,12,"[칼슘, 철, 마그네슘, 인, 칼륨, 나트륨, 아연, 구리, 망간, 셀레늄, 몰리브..."
1,비타민 Vitamins,33,"[비타민 A, 레티놀, 베타카로틴, 티아민, 리보플라빈, 니아신, 니아신당량(NE)..."
2,식염상당량,1,[식염상당량]
3,식품정보,5,"[DB10.4 색인, 10개정 책자 색인, 식품군, 식품명, 출처]"
4,아미노산 Amino acids,21,"[총 아미노산, 총 필수 아미노산, 이소류신, 류신, 라이신, 메티오닌, 페닐알라닌..."
5,일반성분 Proximates,16,"[에너지, 수분, 단백질, 지방, 회분, 탄수화물, 당류, 자당, 포도당, 과당, ..."
6,지방산 Fatty acids,47,"[총 지방산, 총 필수 지방산, 총 포화 지방산, 부티르산 (4:0), 카프로산 (..."
7,콜레스테롤,1,[콜레스테롤]
8,폐기율,1,[폐기율]


## 10. 수치형 컬럼 전체 정제

In [12]:
# 원본 보존
nutrition_clean_df = nutrition_df.copy()

# 원본 값의 상태정보 저장
nutrition_status_df = pd.DataFrame(
    index=nutrition_df.index
)


for col in value_cols:
    parsed = nutrition_df[col].map(parse_nutrition_value)

    # 계산에 사용할 숫자값
    nutrition_clean_df[col] = parsed.map(
        lambda x: x[0]
    )

    # 값의 원래 상태
    nutrition_status_df[col] = parsed.map(
        lambda x: x[1]
    )


print("전처리 대상 컬럼 :", len(value_cols))
print("정제 데이터 shape :", nutrition_clean_df.shape)


print("\n=== 값 상태 집계 ===")

status_counts = (
    nutrition_status_df
    .stack()
    .value_counts()
)

display(status_counts)

전처리 대상 컬럼 : 132
정제 데이터 shape : (3366, 137)

=== 값 상태 집계 ===


numeric                      305480
unmeasured                   133405
quoted_or_converted            4832
trace                           431
quoted_or_converted_trace       164
Name: count, dtype: int64

## 결측치 

In [13]:
missing_report = pd.DataFrame({
    "column": value_cols,
    "missing_count": [
        nutrition_clean_df[col].isna().sum()
        for col in value_cols
    ]
})

missing_report["missing_rate"] = (
    missing_report["missing_count"]
    / len(nutrition_clean_df)
    * 100
).round(2)

missing_report = missing_report.sort_values(
    "missing_rate",
    ascending=False
).reset_index(drop=True)

display(missing_report)

,column,missing_count,missing_rate
0,비타민 B6,1777,52.79
1,타우린,1668,49.55
2,비타민 K2,1607,47.74
3,니코틴산,1502,44.62
4,니코틴아미드,1502,44.62
...,...,...,...
127,지방,18,0.53
128,회분,12,0.36
129,수분,9,0.27
130,단백질,1,0.03


## 중복 검증

In [14]:
print("전체 식품 수 :", len(nutrition_clean_df))

print(
    "DB10.4 색인 중복 :",
    nutrition_clean_df["DB10.4 색인"].duplicated().sum()
)

print(
    "완전 중복 행 :",
    nutrition_clean_df.duplicated().sum()
)

print(
    "식품명 중복 :",
    nutrition_clean_df["식품명"].duplicated().sum()
)

전체 식품 수 : 3366
DB10.4 색인 중복 : 0
완전 중복 행 : 0
식품명 중복 : 0


## 이상치

In [15]:
outlier_rows = []

for col in value_cols:
    series = nutrition_clean_df[col].dropna()

    if series.empty:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    outlier_count = (
        (series < lower) |
        (series > upper)
    ).sum()

    outlier_rows.append({
        "column": col,
        "min": series.min(),
        "max": series.max(),
        "Q1": q1,
        "Q3": q3,
        "IQR_outlier_count": outlier_count
    })

outlier_report = pd.DataFrame(outlier_rows)

display(
    outlier_report.sort_values(
        "IQR_outlier_count",
        ascending=False
    )
)

,column,min,max,Q1,Q3,IQR_outlier_count
131,폐기율,0.0,91.00,0.00,5.0000,692
122,도코사 펜타에노산 (22:5(n-3)),0.0,2991.00,0.00,0.4175,566
104,미리스톨레산 (14:1),0.0,687.67,0.00,0.0000,539
30,베타카로틴,0.0,47375.00,0.00,98.7500,537
110,에루크산 (22:1),0.0,6231.61,0.00,0.3100,513
...,...,...,...,...,...,...
43,엽산_ 합성 엽산,0.0,523.00,0.00,0.0000,34
62,총 필수 아미노산,0.0,40550.00,437.50,6611.0000,34
64,류신,0.0,7300.00,69.25,1135.0000,31
70,발린,0.0,5400.00,52.00,723.5000,31


In [16]:
# 모든 수치형 컬럼의 음수값 확인
negative_report = pd.DataFrame({
    "column": value_cols,
    "negative_count": [
        (nutrition_clean_df[col] < 0).sum()
        for col in value_cols
    ]
})

negative_report = negative_report[
    negative_report["negative_count"] > 0
]

print("=== 음수값 존재 컬럼 ===")
display(negative_report)


# 가식부 100g 기준 물리적 범위 검사
physical_rows = []

for _, row in nutrition_meta_df.iterrows():

    col = row["column"]
    unit = row["unit"]

    if col not in value_cols:
        continue

    series = nutrition_clean_df[col]

    # g / 100g 또는 % 값은 0~100 범위
    if unit in ["g", "%"]:
        lower = 0
        upper = 100

    # 에너지는 별도 범위
    elif col == "에너지":
        lower = 0
        upper = 1000

    else:
        continue

    invalid_mask = (
        (series < lower) |
        (series > upper)
    )

    physical_rows.append({
        "column": col,
        "unit": unit,
        "lower_bound": lower,
        "upper_bound": upper,
        "invalid_count": invalid_mask.sum(),
        "min": series.min(),
        "max": series.max()
    })


physical_outlier_report = pd.DataFrame(physical_rows)

print("\n=== 물리적 범위 이상값 ===")

display(
    physical_outlier_report.sort_values(
        "invalid_count",
        ascending=False
    )
)

=== 음수값 존재 컬럼 ===


,column,negative_count



=== 물리적 범위 이상값 ===


,column,unit,lower_bound,upper_bound,invalid_count,min,max
25,식염상당량,g,0,100,2,0.0,103.20
1,수분,g,0,100,0,0.0,100.00
0,에너지,kcal,0,1000,0,0.0,921.00
3,지방,g,0,100,0,0.0,100.00
4,회분,g,0,100,0,0.0,99.78
5,탄수화물,g,0,100,0,0.0,99.95
6,당류,g,0,100,0,0.0,99.90
7,자당,g,0,100,0,0.0,99.90
8,포도당,g,0,100,0,0.0,85.50
9,과당,g,0,100,0,0.0,99.90


## 컬럼 선별

In [22]:
# 공식 총량 컬럼 확인
total_cols_df = nutrition_meta_df[
    nutrition_meta_df["column"].str.startswith("총 ", na=False)
][
    ["group", "column", "unit"]
].reset_index(drop=True)

display(total_cols_df)

,group,column,unit
0,일반성분 Proximates,총 식이섬유,g
1,아미노산 Amino acids,총 아미노산,mg
2,아미노산 Amino acids,총 필수 아미노산,mg
3,지방산 Fatty acids,총 지방산,g
4,지방산 Fatty acids,총 필수 지방산,g
5,지방산 Fatty acids,총 포화 지방산,g
6,지방산 Fatty acids,총 불포화 지방산,g
7,지방산 Fatty acids,총 단일 불포화지방산,g
8,지방산 Fatty acids,총 다가 불포화지방산,g
9,지방산 Fatty acids,총 트랜스 지방산,g


In [23]:
total_summary_df = (
    nutrition_meta_df
    .groupby("group")
    .agg(
        전체컬럼수=("column", "count"),
        총량컬럼=("column", lambda x: [
            col for col in x
            if str(col).startswith("총 ")
        ])
    )
    .reset_index()
)

total_summary_df["총량존재"] = (
    total_summary_df["총량컬럼"]
    .apply(len)
    .gt(0)
)

display(total_summary_df)

,group,전체컬럼수,총량컬럼,총량존재
0,무기질 Minerals,12,[],False
1,비타민 Vitamins,33,[],False
2,식염상당량,1,[],False
3,식품정보,5,[],False
4,아미노산 Amino acids,21,"[총 아미노산, 총 필수 아미노산]",True
5,일반성분 Proximates,16,[총 식이섬유],True
6,지방산 Fatty acids,47,"[총 지방산, 총 필수 지방산, 총 포화 지방산, 총 불포화 지방산, 총 단일 불포...",True
7,콜레스테롤,1,[],False
8,폐기율,1,[],False


In [26]:
# 수치형 컬럼 결측률 분포 확인
missing_stats = missing_report["missing_rate"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]
)

display(missing_stats)

count    132.000000
mean      30.158333
std       11.666233
min        0.000000
25%       29.417500
50%       31.250000
75%       37.700000
90%       40.008000
max       52.790000
Name: missing_rate, dtype: float64

In [27]:
q1 = missing_report["missing_rate"].quantile(0.25)
q3 = missing_report["missing_rate"].quantile(0.75)

iqr = q3 - q1

missing_threshold = q3 + 1.5 * iqr

print("Q1 :", round(q1, 2))
print("Q3 :", round(q3, 2))
print("IQR :", round(iqr, 2))
print("고결측 컬럼 기준 :", round(missing_threshold, 2), "%")


high_missing_cols = missing_report[
    missing_report["missing_rate"] > missing_threshold
].copy()

display(high_missing_cols)

Q1 : 29.42
Q3 : 37.7
IQR : 8.28
고결측 컬럼 기준 : 50.12 %


,column,missing_count,missing_rate
0,비타민 B6,1777,52.79


In [28]:
coverage_rows = []

for col in value_cols:
    series = nutrition_clean_df[col]

    total = len(series)
    missing_count = series.isna().sum()
    zero_count = (series == 0).sum()
    positive_count = (series > 0).sum()

    coverage_rows.append({
        "column": col,
        "group": nutrition_meta_df.loc[
            nutrition_meta_df["column"] == col,
            "group"
        ].iloc[0],

        "missing_rate": round(missing_count / total * 100, 2),
        "zero_rate": round(zero_count / total * 100, 2),
        "positive_rate": round(positive_count / total * 100, 2),
        "available_rate": round(
            (total - missing_count) / total * 100,
            2
        )
    })


coverage_report = pd.DataFrame(coverage_rows)

display(
    coverage_report.sort_values(
        "positive_rate"
    )
)

,column,group,missing_rate,zero_rate,positive_rate,available_rate
43,엽산_ 합성 엽산,비타민 Vitamins,40.46,58.53,1.01,59.54
12,갈락토오스,일반성분 Proximates,38.80,59.54,1.66,61.20
47,비타민 D2,비타민 Vitamins,40.61,57.37,2.02,59.39
10,유당,일반성분 Proximates,37.70,58.88,3.42,62.30
86,부티르산 (4:0),지방산 Fatty acids,34.88,61.02,4.10,65.12
...,...,...,...,...,...,...
16,칼슘,무기질 Minerals,0.89,1.75,97.36,99.11
2,단백질,일반성분 Proximates,0.03,2.44,97.53,99.97
4,회분,일반성분 Proximates,0.36,1.22,98.43,99.64
1,수분,일반성분 Proximates,0.27,0.68,99.05,99.73


In [29]:
# positive_rate 분포 확인
positive_stats = coverage_report["positive_rate"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9]
)

display(positive_stats)


# 하위 25% 경계를 저커버리지 후보 기준으로 사용
positive_threshold = coverage_report[
    "positive_rate"
].quantile(0.25)

print(
    "저커버리지 후보 기준 :",
    round(positive_threshold, 2),
    "%"
)


low_coverage_cols = coverage_report[
    coverage_report["positive_rate"] <= positive_threshold
].sort_values("positive_rate")

display(low_coverage_cols)

count    132.000000
mean      45.457348
std       27.239718
min        1.010000
25%       21.977500
50%       46.805000
75%       65.950000
90%       79.462000
max       99.580000
Name: positive_rate, dtype: float64

저커버리지 후보 기준 : 21.98 %


,column,group,missing_rate,zero_rate,positive_rate,available_rate
43,엽산_ 합성 엽산,비타민 Vitamins,40.46,58.53,1.01,59.54
12,갈락토오스,일반성분 Proximates,38.80,59.54,1.66,61.20
47,비타민 D2,비타민 Vitamins,40.61,57.37,2.02,59.39
10,유당,일반성분 Proximates,37.70,58.88,3.42,62.30
86,부티르산 (4:0),지방산 Fatty acids,34.88,61.02,4.10,65.12
106,헵타데센산 (17:1),지방산 Fatty acids,34.82,60.90,4.28,65.18
87,카프로산 (6:0),지방산 Fatty acids,34.82,60.34,4.84,65.18
115,감마 리놀렌산 (18:3 (n-6)),지방산 Fatty acids,35.80,59.21,4.99,64.20
91,트라이데칸산 (13:0),지방산 Fatty acids,39.28,55.35,5.38,60.72
48,비타민 D3,비타민 Vitamins,39.33,55.20,5.47,60.67


In [30]:
# 데이터 분포에서 기준 자동 산출
missing_q3 = coverage_report[
    "missing_rate"
].quantile(0.75)

positive_q1 = coverage_report[
    "positive_rate"
].quantile(0.25)


quality_exclude_candidates = coverage_report[
    (coverage_report["missing_rate"] >= missing_q3)
    &
    (coverage_report["positive_rate"] <= positive_q1)
].sort_values(
    ["positive_rate", "missing_rate"],
    ascending=[True, False]
)


print("결측률 상위 25% 기준 :", round(missing_q3, 2), "%")
print("양수값 하위 25% 기준 :", round(positive_q1, 2), "%")
print("제외 후보 컬럼 수 :", len(quality_exclude_candidates))

display(quality_exclude_candidates)

결측률 상위 25% 기준 : 37.7 %
양수값 하위 25% 기준 : 21.98 %
제외 후보 컬럼 수 : 15


,column,group,missing_rate,zero_rate,positive_rate,available_rate
43,엽산_ 합성 엽산,비타민 Vitamins,40.46,58.53,1.01,59.54
12,갈락토오스,일반성분 Proximates,38.80,59.54,1.66,61.20
47,비타민 D2,비타민 Vitamins,40.61,57.37,2.02,59.39
10,유당,일반성분 Proximates,37.70,58.88,3.42,62.30
91,트라이데칸산 (13:0),지방산 Fatty acids,39.28,55.35,5.38,60.72
48,비타민 D3,비타민 Vitamins,39.33,55.20,5.47,60.67
55,베타 토코트리에놀,비타민 Vitamins,39.13,55.17,5.70,60.87
60,비타민 K2,비타민 Vitamins,47.74,45.93,6.33,52.26
57,델타 토코트리에놀,비타민 Vitamins,39.13,54.28,6.60,60.87
117,디호모 리놀렌산 (20:3(n-3)),지방산 Fatty acids,38.77,53.71,7.52,61.23


In [40]:
# 1차 Feature Selection
# 세부 아미노산/지방산은 대표 지표만 유지하고,
# 데이터 품질이 낮은 컬럼 제외

detail_groups = [
    "아미노산 Amino acids",
    "지방산 Fatty acids"
]

# 아미노산 대표 지표
amino_keep_cols = [
    "총 아미노산",
    "총 필수 아미노산"
]

# 지방산 대표 지표
fatty_keep_cols = [
    "총 필수 지방산",
    "총 포화 지방산",
    "총 불포화 지방산",
    "오메가3 지방산",
    "오메가6 지방산",
    "총 트랜스 지방산"
]

# 아미노산/지방산 외 그룹은 우선 유지
base_cols = nutrition_meta_df.loc[
    ~nutrition_meta_df["group"].isin(detail_groups),
    "column"
].tolist()

# 실제 DB에 존재하는 대표 컬럼만 추가
summary_cols = [
    col
    for col in amino_keep_cols + fatty_keep_cols
    if col in nutrition_clean_df.columns
]

# 데이터 품질 기준 제외
quality_exclude_cols = set(
    quality_exclude_candidates["column"]
)

selected_cols = [
    col
    for col in base_cols + summary_cols
    if col not in quality_exclude_cols
]

nutrition_selected_df = nutrition_clean_df[
    selected_cols
].copy()

print("전체 컬럼 :", nutrition_clean_df.shape[1])
print("1차 선별 컬럼 :", nutrition_selected_df.shape[1])

전체 컬럼 : 137
1차 선별 컬럼 : 67


In [41]:
# LLM 급식 추천용 핵심 영양 지표
# 대상: 유아·학생·성인·노인·질환별 식단 조건을 폭넓게 고려

llm_feature_cols = [
    # 식품 정보
    "DB10.4 색인",
    "식품군",
    "식품명",
    "출처",

    # 기본 영양 / 체중 / 당뇨 / 성장 / 영양불량
    "에너지",
    "수분",
    "단백질",
    "지방",
    "탄수화물",
    "당류",
    "총 식이섬유",

    # 비타민
    "비타민 A",       # 성장, 시각, 학교급식
    "비타민 D",       # 성장, 골건강, 노인
    "비타민 C",       # 학교급식, 철 흡수 등
    "티아민",          # 비타민 B1, 학교급식
    "리보플라빈",      # 비타민 B2, 학교급식
    # "엽산",            # 임신, 조혈
    "비타민 B12",     # 조혈, 노인
    "비타민 K1",      # 혈액응고 관련 특수 식단

    # 무기질
    "칼슘",            # 성장, 골건강
    "철",              # 성장, 빈혈
    "마그네슘",        # 골·근육
    "인",              # 골건강 + 신장질환
    "칼륨",            # 혈압 + 신장질환
    "나트륨",          # 저염, 고혈압, 신장질환
    "아연",            # 성장, 면역, 회복
    "요오드",          # 갑상선

    # 단백질 질 보조 지표
    "총 필수 아미노산",

    # 지방의 질
    "총 포화 지방산",
    "총 불포화 지방산",
    "오메가3 지방산",
    "오메가6 지방산",
    "총 트랜스 지방산",

    # 사용자 질의 대응용 보조 지표
    "콜레스테롤"
]

# 1차 선별된 67개 중에서만 선택
final_cols = [
    col for col in llm_feature_cols
    if col in nutrition_selected_df.columns
]

# 혹시 1차 전처리에서 이미 빠졌거나
# DB 이름이 다른 컬럼 확인
missing_target_cols = [
    col for col in llm_feature_cols
    if col not in nutrition_selected_df.columns
]

# LLM용 최종 영양 데이터
llm_nutrition_df = nutrition_selected_df[
    final_cols
].copy()

print("1차 선별 컬럼 :", nutrition_selected_df.shape[1])
print("LLM 최종 컬럼 :", llm_nutrition_df.shape[1])

print("\n최종 컬럼:")
print(final_cols)

print("\n찾지 못한 컬럼:")
print(missing_target_cols)

1차 선별 컬럼 : 67
LLM 최종 컬럼 : 33

최종 컬럼:
['DB10.4 색인', '식품군', '식품명', '출처', '에너지', '수분', '단백질', '지방', '탄수화물', '당류', '총 식이섬유', '비타민 A', '비타민 D', '비타민 C', '티아민', '리보플라빈', '비타민 B12', '비타민 K1', '칼슘', '철', '마그네슘', '인', '칼륨', '나트륨', '아연', '요오드', '총 필수 아미노산', '총 포화 지방산', '총 불포화 지방산', '오메가3 지방산', '오메가6 지방산', '총 트랜스 지방산', '콜레스테롤']

찾지 못한 컬럼:
[]


## 식품명 정규화

In [42]:
def normalize_food_name(name):
    if pd.isna(name):
        return ""

    name = str(name).strip()

    # 전각 쉼표 등 기본 표기 통일
    name = name.replace("，", ",")

    # 띄어쓰기 차이 제거
    name = re.sub(r"\s+", "", name)

    # 영문 포함 시 대소문자 차이 제거
    name = name.casefold()

    return name


# 원본은 그대로 보존하고 매칭용 컬럼만 추가
ingredient_df["food_name_norm"] = (
    ingredient_df["ingredient_food_Nm"]
    .map(normalize_food_name)
)

llm_nutrition_df = llm_nutrition_df.copy()

llm_nutrition_df["food_name_norm"] = (
    llm_nutrition_df["식품명"]
    .map(normalize_food_name)
)

print("MenuGen 식재료 행 :", len(ingredient_df))
print("MenuGen 고유 식재료명 :", ingredient_df["food_name_norm"].nunique())
print("영양 DB 식품 수 :", len(llm_nutrition_df))
print("영양 DB 정규화명 중복 :", llm_nutrition_df["food_name_norm"].duplicated().sum())

MenuGen 식재료 행 : 22582
MenuGen 고유 식재료명 : 1388
영양 DB 식품 수 : 3366
영양 DB 정규화명 중복 : 0


In [ ]:
# 식품정보를 제외한 실제 영양성분 컬럼
info_cols = [
    "DB10.4 색인",
    "식품군",
    "식품명",
    "출처"
]

nutrient_cols = [
    col for col in final_cols
    if col not in info_cols
]

# MenuGen 식재료 ↔ 영양 DB 연결
ingredient_nutrition_df = ingredient_df.merge(
    llm_nutrition_df[
        ["food_name_norm"] + nutrient_cols
    ],
    on="food_name_norm",
    how="left"
)

# 100g 기준 영양값 → 실제 사용 중량 기준으로 환산
for col in nutrient_cols:
    ingredient_nutrition_df[col] = (
        ingredient_nutrition_df[col]
        * ingredient_nutrition_df["ingredient_food_Wgh"]
        / 100
    )

연결된 식재료 행 : 22582


In [45]:
menu_nutrition_df = (
    ingredient_nutrition_df
    .groupby(
        ["menu_fd_Code", "menu_fd_Nm"],
        as_index=False
    )[nutrient_cols]
    .sum(min_count=1)
)

print("메뉴별 영양 데이터 :", menu_nutrition_df.shape)

display(menu_nutrition_df.head())

메뉴별 영양 데이터 : (3250, 31)


,menu_fd_Code,menu_fd_Nm,에너지,수분,단백질,지방,탄수화물,당류,총 식이섬유,비타민 A,...,나트륨,아연,요오드,총 필수 아미노산,총 포화 지방산,총 불포화 지방산,오메가3 지방산,오메가6 지방산,총 트랜스 지방산,콜레스테롤
0,D011001,눌은밥,303.6,366.96,5.808,0.440,66.352,0.176,2.64,0.0,...,57.2,1.496,0.0,2068.0,0.176,0.220,0.000,0.132,0.0,0.0
1,D011002,쌀밥,319.2,134.40,5.481,0.798,69.006,0.021,0.42,0.0,...,2.1,1.365,0.0,1881.6,0.273,0.378,0.000,0.231,0.0,0.0
2,D011003,찰밥,325.5,132.51,5.418,0.693,71.253,0.231,0.42,0.0,...,2.1,1.470,0.0,1688.4,0.168,0.273,0.000,0.147,0.0,0.0
3,D011004,현미밥,348.6,125.16,7.077,2.268,74.193,0.840,2.73,0.0,...,2.1,2.310,0.0,2280.6,0.504,1.176,0.021,0.609,0.0,0.0
4,D011006,현미밥,265.6,95.36,5.392,1.728,56.528,0.640,2.08,0.0,...,1.6,1.760,0.0,1737.6,0.384,0.896,0.016,0.464,0.0,0.0


In [ ]:
menu_allergy_df = (
    ingredient_df
    .groupby(
        ["menu_fd_Code", "menu_fd_Nm"],
        as_index=False
    )["ingredient_allrgy_Info"]
    .agg(
        lambda x: ", ".join(
            sorted(
                set(
                    str(v).strip()
                    for v in x
                    if pd.notna(v)
                    and str(v).strip()
                )
            )
        )
    )
)

menu_allergy_df = menu_allergy_df.rename(
    columns={
        "ingredient_allrgy_Info": "allergy_info"
    }
)

print(
    "메뉴별 알레르기 데이터 :",
    menu_allergy_df.shape
)

메뉴별 알레르기 데이터 : (3250, 3)


In [98]:
menu_rag_df = menu_nutrition_df.merge(
    menu_allergy_df,
    on=[
        "menu_fd_Code",
        "menu_fd_Nm"
    ],
    how="left"
)

print(
    "최종 메뉴 데이터 :",
    menu_rag_df.shape
)

display(
    menu_rag_df[
        [
            "menu_fd_Code",
            "menu_fd_Nm",
            "allergy_info"
        ]
    ].head()
)

최종 메뉴 데이터 : (3250, 32)


,menu_fd_Code,menu_fd_Nm,allergy_info
0,D011001,눌은밥,
1,D011002,쌀밥,
2,D011003,찰밥,
3,D011004,현미밥,
4,D011006,현미밥,


In [99]:
menu_meta_df = menu_df[
    [
        "fd_Code",
        "upper_Fd_Grupp_Nm",
        "fd_Grupp_Nm"
    ]
].copy()

menu_meta_df = menu_meta_df.rename(
    columns={
        "fd_Code": "menu_fd_Code"
    }
)

menu_rag_df = menu_rag_df.merge(
    menu_meta_df,
    on="menu_fd_Code",
    how="left"
)

print(
    menu_rag_df[
        [
            "menu_fd_Nm",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ].head()
)

  menu_fd_Nm upper_Fd_Grupp_Nm fd_Grupp_Nm
0        눌은밥                밥류          쌀밥
1         쌀밥                밥류          쌀밥
2         찰밥                밥류          쌀밥
3        현미밥                밥류          쌀밥
4        현미밥                밥류          쌀밥


In [100]:
display(
    menu_rag_df[
        [
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    )
)

,upper_Fd_Grupp_Nm,fd_Grupp_Nm
2869,과일 및 과일가공품,가공품류
2776,과일 및 과일가공품,과일류
456,과자 및 빵류,과자류
388,과자 및 빵류,빵류
1741,구이류,기타 구이류
...,...,...
2222,튀김류,채소류튀김
2254,튀김류,해조류튀김
2511,회류,어패류회
2521,회류,육류회


In [101]:
category_summary = (
    menu_rag_df
    .groupby("upper_Fd_Grupp_Nm")
    .agg(
        menu_count=("menu_fd_Code", "count")
    )
    .sort_index()
)

display(category_summary)

,menu_count
upper_Fd_Grupp_Nm,
과일 및 과일가공품,134
과자 및 빵류,146
구이류,142
국(탕)류,605
김치류,62
떡류,41
면 및 만두류,121
무침류,188
밥류,388


In [109]:
# 추천 대상에서 제외할 명확한 대분류
NON_MENU_GROUPS = {
    "원재료",
    "장류",
    "주류"
}

# 분류상 메뉴군에 들어가 있지만 실제 추천 메뉴로는 부적절한 예외
EXCLUDED_MENU_NAMES = {
    "육수(소고기)"
}

recommendable_menu_df = (
    menu_rag_df[
        ~menu_rag_df["upper_Fd_Grupp_Nm"]
        .isin(NON_MENU_GROUPS)
        &
        ~menu_rag_df["menu_fd_Nm"]
        .isin(EXCLUDED_MENU_NAMES)
    ]
    .copy()
    .reset_index(drop=True)
)

print("전체 MenuGen :", len(menu_rag_df))
print("추천 가능 메뉴 :", len(recommendable_menu_df))

전체 MenuGen : 3250
추천 가능 메뉴 : 2942


In [106]:
problem_names = [
    "잣(볶은것)",
    "잣(생것)",
    "은행(볶은것)",
    "은행(생것)",
    "육수(소고기)"
]

remaining_problem_df = (
    recommendable_menu_df[
        recommendable_menu_df["menu_fd_Nm"]
        .isin(problem_names)
    ][
        [
            "menu_fd_Nm",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
)

display(remaining_problem_df)

print(
    "추천 후보에 남아있는 문제 항목 :",
    remaining_problem_df["menu_fd_Nm"].tolist()
)

,menu_fd_Nm,upper_Fd_Grupp_Nm,fd_Grupp_Nm


추천 후보에 남아있는 문제 항목 : []


In [107]:
# 추천 후보 중 비메뉴성 항목 의심 목록 확인
# ※ 이 단계에서는 제거하지 않고 검토만 한다.

SUSPICIOUS_NAME_PATTERN = (
    r"육수|"
    r"생것|말린것|건조|"
    r"분말|가루|"
    r"소스|양념|"
    r"원액|농축액"
)

suspicious_menu_df = (
    recommendable_menu_df[
        recommendable_menu_df["menu_fd_Nm"]
        .str.contains(
            SUSPICIOUS_NAME_PATTERN,
            regex=True,
            na=False
        )
    ][
        [
            "menu_fd_Code",
            "menu_fd_Nm",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
    .sort_values(
        [
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm",
            "menu_fd_Nm"
        ]
    )
)

print(
    "비메뉴성 의심 항목 수 :",
    len(suspicious_menu_df)
)

display(
    suspicious_menu_df
)

비메뉴성 의심 항목 수 : 86


,menu_fd_Code,menu_fd_Nm,upper_Fd_Grupp_Nm,fd_Grupp_Nm
2835,D212043,딸기(말린것),과일 및 과일가공품,가공품류
2841,D212049,바나나(말린것),과일 및 과일가공품,가공품류
2847,D212055,블루베리(말린것),과일 및 과일가공품,가공품류
2853,D212061,산수유(말린것),과일 및 과일가공품,가공품류
2854,D212062,살구(말린것),과일 및 과일가공품,가공품류
...,...,...,...,...
1572,D072025,마늘소스수육,찜류,육류찜
1600,D073017,맛양념장을 곁들인 애호박찜,찜류,채소류찜
2164,D121011,동태튀김(녹말가루),튀김류,어패류튀김
2211,D122013,양념통닭,튀김류,육류튀김


In [108]:
# 어떤 키워드 때문에 의심 항목으로 잡혔는지 분류

SUSPICIOUS_KEYWORDS = [
    "육수",
    "생것",
    "말린것",
    "건조",
    "분말",
    "가루",
    "소스",
    "양념",
    "원액",
    "농축액"
]

def find_suspicious_reason(name):
    name = str(name)

    matched = [
        keyword
        for keyword in SUSPICIOUS_KEYWORDS
        if keyword in name
    ]

    return ", ".join(matched)


suspicious_menu_df = suspicious_menu_df.copy()

suspicious_menu_df["reason"] = (
    suspicious_menu_df["menu_fd_Nm"]
    .map(find_suspicious_reason)
)

display(
    suspicious_menu_df[
        [
            "reason",
            "upper_Fd_Grupp_Nm",
            "fd_Grupp_Nm"
        ]
    ]
    .value_counts()
    .reset_index(name="count")
    .sort_values(
        ["reason", "count"],
        ascending=[True, False]
    )
)

,reason,upper_Fd_Grupp_Nm,fd_Grupp_Nm,count
0,가루,음료류,차류,9
4,가루,국(탕)류,된장국류,5
6,가루,국(탕)류,맑은국류,4
7,가루,무침류,숙채,4
13,가루,국(탕)류,냉국류,2
15,가루,밥류,김(초)밥,2
19,가루,구이류,육류구이,1
24,가루,무침류,어패류무침,1
26,가루,음료류,기타 음료류,1
31,가루,튀김류,어패류튀김,1


In [52]:
def extract_user_conditions(user_query):

    response = client.responses.create(
        model="gpt-5.6-luna",
        reasoning={
            "effort": "low"
        },

        input=f"""
너는 급식 메뉴 추천 시스템의 조건 추출기다.

사용자의 요청에서 실제 메뉴 검색과 영양 랭킹에 필요한 조건만 추출해라.

규칙:
- target: 급식 대상
- allergies: 알레르기 목록
- nutrition_high: 많이 섭취하고 싶은 영양소
- nutrition_low: 적게 섭취하고 싶은 영양소
- diseases: 질환 또는 건강 상태
- keywords: 위 항목에 포함되지 않는 실제 추천 조건

'급식', '메뉴', '추천', '음식'처럼
추천 시스템 자체를 설명하는 일반적인 단어는 keywords에 넣지 마라.

사용자 요청:
{user_query}
""",

        text={
            "format": {
                "type": "json_schema",
                "name": "meal_conditions",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "target": {
                            "type": ["string", "null"]
                        },
                        "allergies": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "nutrition_high": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "nutrition_low": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "diseases": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "keywords": {
                            "type": "array",
                            "items": {"type": "string"}
                        }
                    },
                    "required": [
                        "target",
                        "allergies",
                        "nutrition_high",
                        "nutrition_low",
                        "diseases",
                        "keywords"
                    ],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(response.output_text)

In [53]:
test_query = """
고혈압 있는 어르신 급식이야.
대두 알레르기가 있고 나트륨은 낮고
단백질은 충분한 메뉴 추천해줘.
"""

conditions = extract_user_conditions(test_query)

print(json.dumps(
    conditions,
    ensure_ascii=False,
    indent=2
))

{
  "target": "어르신",
  "allergies": [
    "대두"
  ],
  "nutrition_high": [
    "단백질"
  ],
  "nutrition_low": [
    "나트륨"
  ],
  "diseases": [
    "고혈압"
  ],
  "keywords": []
}


In [54]:
def filter_allergies(df, allergies):
    result = df.copy()

    for allergy in allergies:
        mask = (
            result["allergy_info"]
            .fillna("")
            .str.contains(
                str(allergy),
                case=False,
                regex=False
            )
        )

        result = result[~mask]

    return result


def rank_menus(
    df,
    nutrition_high,
    nutrition_low,
    top_k=10
):
    result = df.copy()

    # 실제 데이터에 존재하는 영양소만 사용
    high_cols = [
        col for col in nutrition_high
        if col in result.columns
    ]

    low_cols = [
        col for col in nutrition_low
        if col in result.columns
    ]

    ranking_cols = list(
        dict.fromkeys(high_cols + low_cols)
    )

    # 요청된 영양소가 없으면 점수 계산 없이 반환
    if not ranking_cols:
        result["nutrition_score"] = 0.0
        return result.head(top_k)

    # 요청 영양소 값이 없는 메뉴는 랭킹에서 제외
    result = result.dropna(
        subset=ranking_cols
    ).copy()

    score_cols = []

    # 높을수록 좋은 영양소
    for col in high_cols:
        score_col = f"_high_{col}"

        result[score_col] = (
            result[col]
            .rank(pct=True)
        )

        score_cols.append(score_col)

    # 낮을수록 좋은 영양소
    for col in low_cols:
        score_col = f"_low_{col}"

        result[score_col] = (
            1
            - result[col].rank(pct=True)
        )

        score_cols.append(score_col)

    # 조건별 점수 평균
    result["nutrition_score"] = (
        result[score_cols]
        .mean(axis=1)
    )

    result = (
        result
        .sort_values(
            "nutrition_score",
            ascending=False
        )
        .head(top_k)
    )

    return result

## 메뉴를 RAG Document로 변환

In [110]:
# 메뉴별 식재료 목록 생성
def join_unique_ingredients(values):
    return ", ".join(
        dict.fromkeys(
            str(v).strip()
            for v in values
            if pd.notna(v) and str(v).strip()
        )
    )


ingredient_text_map = (
    ingredient_df
    .groupby("menu_fd_Code")["ingredient_food_Nm"]
    .agg(join_unique_ingredients)
    .to_dict()
)


# 추천 가능한 메뉴 데이터에 식재료 정보 추가
recommendable_menu_df = recommendable_menu_df.copy()

recommendable_menu_df["ingredient_text"] = (
    recommendable_menu_df["menu_fd_Code"]
    .map(ingredient_text_map)
    .fillna("")
)


def safe_text(value, default="없음"):
    if pd.isna(value):
        return default

    value = str(value).strip()

    return value if value else default


# BGE-M3 검색용 메뉴 문서 생성
def make_menu_document(row):

    lines = [
        f"메뉴명: {row['menu_fd_Nm']}",
        f"메뉴 대분류: {safe_text(row['upper_Fd_Grupp_Nm'])}",
        f"메뉴 소분류: {safe_text(row['fd_Grupp_Nm'])}",
        f"식재료: {safe_text(row['ingredient_text'])}"
    ]

    return "\n".join(lines)


menu_documents_df = recommendable_menu_df[
    [
        "menu_fd_Code",
        "menu_fd_Nm",
        "upper_Fd_Grupp_Nm",
        "fd_Grupp_Nm",
        "ingredient_text"
    ]
].copy()

menu_documents_df["document"] = (
    recommendable_menu_df.apply(
        make_menu_document,
        axis=1
    )
)

print("RAG 문서 수 :", len(menu_documents_df))
print()
print(menu_documents_df["document"].iloc[0])

RAG 문서 수 : 2942

메뉴명: 눌은밥
메뉴 대분류: 밥류
메뉴 소분류: 쌀밥
식재료: 즉석밥, 누룽지, 끓는물 부음


In [59]:
# BGE-M3 모델 로드
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import faiss

BGE_MODEL_NAME = "BAAI/bge-m3"

bge_model = BGEM3FlagModel(
    BGE_MODEL_NAME,
    use_fp16=False
)

print("BGE-M3 로드 완료")

d:\script\LLM-PRJ\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\script\LLM-PRJ\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alstj\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: http

BGE-M3 로드 완료


In [111]:
menu_texts = menu_documents_df[
    "document"
].tolist()

output = bge_model.encode(
    menu_texts,
    batch_size=8,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

menu_embeddings = np.asarray(
    output["dense_vecs"],
    dtype="float32"
)

print(
    "Embedding shape :",
    menu_embeddings.shape
)

Inference Embeddings: 100%|██████████| 368/368 [03:30<00:00,  1.75it/s]

Embedding shape : (2942, 1024)


In [112]:
# RAG 임베딩 저장
rag_dir = Path("rag_store")
rag_dir.mkdir(exist_ok=True)

np.save(
    rag_dir / "bge_m3_menu_embeddings.npy",
    menu_embeddings
)

menu_documents_df.to_csv(
    rag_dir / "menu_documents.csv",
    index=False,
    encoding="utf-8-sig"
)

print("BGE-M3 임베딩 저장 완료")

BGE-M3 임베딩 저장 완료


In [113]:
# 원본 embedding은 유지하고 FAISS용 복사본 사용
menu_vectors = menu_embeddings.copy()

# cosine similarity
faiss.normalize_L2(menu_vectors)

embedding_dim = menu_vectors.shape[1]

menu_index = faiss.IndexFlatIP(
    embedding_dim
)

menu_index.add(menu_vectors)

# FAISS 인덱스 저장
faiss.write_index(
    menu_index,
    str(rag_dir / "menu_index.faiss")
)

print("Embedding dimension :", embedding_dim)
print("FAISS 저장 문서 :", menu_index.ntotal)

Embedding dimension : 1024
FAISS 저장 문서 : 2942


In [152]:
# BGE-M3 Retriever
def retrieve_menus(query, top_k=50):

    query_output = bge_model.encode(
        [query],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        query_output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(query_vector)

    scores, indices = menu_index.search(
        query_vector,
        top_k
    )

    retrieved = (
        recommendable_menu_df.iloc[indices[0]].copy()
    )

    retrieved["retrieval_score"] = scores[0]

    return retrieved

In [153]:
# 한 끼 급식 식단의 기본 구성
MEAL_SLOT_GROUPS = {
    "주식": {
        "밥류",
        "면 및 만두류",
        "죽류"
    },

    "국": {
        "국(탕)류",
        "찌개류"
    },

    "주찬": {
        "구이류",
        "볶음류",
        "조림류",
        "찜류",
        "전류",
        "튀김류"
    },

    "부찬": {
        "무침류",
        "볶음류",
        "조림류",
        "전류",
        "절임류"
    },

    "김치": {
        "김치류"
    }
}

In [155]:
# 식단 슬롯별 FAISS Index 생성
slot_indexes = {}
slot_menu_dfs = {}

for slot_name, allowed_groups in MEAL_SLOT_GROUPS.items():

    # 해당 슬롯에 사용할 수 있는 메뉴 위치
    mask = (
        recommendable_menu_df[
            "upper_Fd_Grupp_Nm"
        ]
        .isin(allowed_groups)
        .to_numpy()
    )

    # 메뉴 dataframe
    slot_df = (
        recommendable_menu_df[
            mask
        ]
        .copy()
        .reset_index(drop=True)
    )

    # 기존 BGE-M3 embedding에서 해당 행만 추출
    slot_vectors = (
        menu_embeddings[
            mask
        ]
        .copy()
    )

    faiss.normalize_L2(
        slot_vectors
    )

    slot_index = faiss.IndexFlatIP(
        slot_vectors.shape[1]
    )

    slot_index.add(
        slot_vectors
    )

    slot_menu_dfs[
        slot_name
    ] = slot_df

    slot_indexes[
        slot_name
    ] = slot_index

    print(
        slot_name,
        ":",
        len(slot_df)
    )

주식 : 558
국 : 818
주찬 : 738
부찬 : 608
김치 : 62


In [156]:
def retrieve_slot_candidates(
    slot_name,
    semantic_query="",
    allergies=None,
    top_k=15
):
    if allergies is None:
        allergies = []

    slot_df = slot_menu_dfs[
        slot_name
    ]

    slot_index = slot_indexes[
        slot_name
    ]

    query = f"{slot_name} 급식 메뉴"

    if semantic_query:
        query += (
            "\n추가 조건: "
            + semantic_query
        )

    # Query Embedding
    query_output = bge_model.encode(
        [query],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        query_output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(
        query_vector
    )

    # 슬롯 내부 전체 후보를 유사도 순으로 검색
    search_k = slot_index.ntotal

    scores, indices = (
        slot_index.search(
            query_vector,
            search_k
        )
    )

    retrieved = (
        slot_df
        .iloc[indices[0]]
        .copy()
    )

    retrieved["retrieval_score"] = (
        scores[0]
    )

    # 알레르기 Hard Filter
    retrieved = filter_allergies(
        retrieved,
        allergies
    )

    # 안전 필터를 통과한 후보 중
    # Semantic similarity 상위 N개
    return (
        retrieved
        .head(top_k)
        .reset_index(drop=True)
    )

In [157]:
semantic_parts = []

if conditions.get("target"):
    semantic_parts.append(
        conditions["target"]
    )

semantic_parts.extend(
    conditions.get(
        "keywords",
        []
    )
)

semantic_query = " ".join(
    semantic_parts
)

print("semantic_query :", semantic_query)

semantic_query : 요양원 어르신


In [208]:
PRE_GATE_POOL_SIZE = {
    "주식": 12,
    "국": 15,
    "주찬": 20,
    "부찬": 25,
    "김치": 10
}

slot_candidates = {}

for slot_name in MEAL_SLOT_GROUPS:

    slot_candidates[slot_name] = retrieve_slot_candidates(
        slot_name=slot_name,
        semantic_query=semantic_query,
        allergies=conditions["allergies"],
        top_k=PRE_GATE_POOL_SIZE[slot_name]
    )

    print(
        slot_name,
        ":",
        len(slot_candidates[slot_name])
    )

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 17.98it/s]


주식 : 12


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 15.01it/s]


국 : 15


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 17.61it/s]


주찬 : 20


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 17.67it/s]


부찬 : 25


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 15.63it/s]

김치 : 10


In [209]:
slot_gate_items = []

for slot_name, df in slot_candidates.items():

    for idx, row in df.iterrows():

        slot_gate_items.append({
            "candidate_id": f"{slot_name}_{idx}",
            "slot": slot_name,
            "menu_name": row["menu_fd_Nm"],
            "upper_group": row["upper_Fd_Grupp_Nm"],
            "sub_group": row["fd_Grupp_Nm"]
        })

print(
    "검사할 슬롯 후보 수 :",
    len(slot_gate_items)
)

검사할 슬롯 후보 수 : 82


In [218]:
def evaluate_slot_candidates(
    slot_gate_items,
    conditions
):

    target_context = " ".join([
        str(conditions.get("target") or ""),
        *conditions.get("keywords", [])
    ]).strip()

    response = client.responses.create(
        model="gpt-5.6-luna",
        reasoning={"effort": "low"},
        input=f"""
너는 급식 식단의 메뉴 역할 적합성을 판정하는 분류기다.

급식 대상:
{target_context}

각 후보가 현재 지정된 슬롯에 실제 급식 식단 구성상
자연스러운 경우에만 keep=true로 판단한다.

[슬롯 기준]

주식:
- 밥, 죽, 면 등 식사의 중심 탄수화물 음식
- 이유식처럼 현재 급식 대상과 명백히 맞지 않으면 제외

국:
- 국, 탕, 찌개 등 국물 음식
- 메뉴명이 국/탕류로 분류되었더라도
  통닭·백숙 등 독립적인 주찬 성격이 강하면 제외

주찬:
- 한 끼의 중심 반찬
- 육류, 생선, 달걀, 두부 등 단백질 중심 또는
  일반적으로 메인 반찬으로 제공되는 음식
- 간식, 디저트, 떡류, 일품식,
  채소 단독 소량 반찬은 제외

부찬:
- 주찬을 보조하는 일반적인 반찬
- 나물, 무침, 조림, 채소반찬 등
- 떡볶이·면·밥처럼 주식성 또는 일품식 성격이 강한 음식,
  간식·디저트는 제외

김치:
- 김치류만 유지

추가 규칙:
- 영양성분을 보고 판단하지 않는다.
- 질환 치료 적합성을 판단하지 않는다.
- 메뉴명과 분류를 근거로 식단 역할만 판단한다.
- 명백히 부자연스러운 슬롯 배치는 제외한다.
- 합리적으로 해당 역할에 제공될 수 있다면 유지한다.
- 메뉴명에 어린이, 유아, 영유아 등 특정 대상이 명시되어 있고 현재 급식 대상과 명백히 다르면 제외한다.

검사 대상:
{json.dumps(
    slot_gate_items,
    ensure_ascii=False,
    indent=2
)}
""",
        text={
            "format": {
                "type": "json_schema",
                "name": "slot_candidate_gate",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "results": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "candidate_id": {
                                        "type": "string"
                                    },
                                    "keep": {
                                        "type": "boolean"
                                    },
                                    "reason": {
                                        "type": "string"
                                    }
                                },
                                "required": [
                                    "candidate_id",
                                    "keep",
                                    "reason"
                                ],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["results"],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(
        response.output_text
    )

In [219]:
slot_gate_result = evaluate_slot_candidates(
    slot_gate_items,
    conditions
)

slot_gate_result

{'results': [{'candidate_id': '주식_0',
   'keep': False,
   'reason': '이유식으로 현재 급식 대상과 맞지 않음'},
  {'candidate_id': '주식_1', 'keep': True, 'reason': '밥류의 식사 중심 음식'},
  {'candidate_id': '주식_2', 'keep': True, 'reason': '밥류의 식사 중심 음식'},
  {'candidate_id': '주식_3', 'keep': True, 'reason': '밥류의 식사 중심 음식'},
  {'candidate_id': '주식_4', 'keep': True, 'reason': '잡곡밥류의 식사 중심 음식'},
  {'candidate_id': '주식_5', 'keep': True, 'reason': '죽류의 식사 중심 음식'},
  {'candidate_id': '주식_6', 'keep': True, 'reason': '죽류의 식사 중심 음식'},
  {'candidate_id': '주식_7', 'keep': True, 'reason': '잡곡밥류의 식사 중심 음식'},
  {'candidate_id': '주식_8', 'keep': True, 'reason': '죽류의 식사 중심 음식'},
  {'candidate_id': '주식_9', 'keep': True, 'reason': '죽류의 식사 중심 음식'},
  {'candidate_id': '주식_10', 'keep': True, 'reason': '죽류의 식사 중심 음식'},
  {'candidate_id': '주식_11', 'keep': True, 'reason': '밥류의 식사 중심 음식'},
  {'candidate_id': '국_0', 'keep': False, 'reason': '백숙으로 독립적인 주찬 성격이 강함'},
  {'candidate_id': '국_1', 'keep': True, 'reason': '국물 음식으로 자연스러움'},
  {'cand

In [200]:
rejected_candidates = [
    item
    for item in slot_gate_result["results"]
    if not item["keep"]
]

for item in rejected_candidates:
    print(
        item["candidate_id"],
        "→",
        item["reason"]
    )

주식_0 → 이유식으로 현재 급식 대상인 어르신의 주식으로 부자연스럽다.
국_0 → 백숙은 국물보다 독립적인 주찬 성격이 강하다.
주찬_2 → 채소 중심의 볶음으로 일반적인 부찬 성격이 강하다.
주찬_3 → 떡볶이는 주식성 또는 간식성 메뉴로 주찬 역할에 부자연스럽다.
주찬_4 → 채소 중심의 전으로 주찬보다는 부찬 성격이 강하다.
주찬_5 → 화전은 떡·간식 및 디저트 성격이 강하다.
주찬_6 → 양송이만으로 구성된 채소성 구이로 부찬 성격이 강하다.
주찬_7 → 콩나물잡채는 채소 중심의 곁들임 반찬으로 주찬 성격이 약하다.
부찬_1 → 떡볶이는 주식성 또는 간식성 메뉴로 부찬에 부자연스럽다.
부찬_13 → 화전은 떡·간식 및 디저트 성격이 강하다.


In [220]:
def apply_slot_gate(
    slot_candidates,
    slot_gate_result
):
    keep_map = {
        item["candidate_id"]: item["keep"]
        for item in slot_gate_result["results"]
    }

    filtered_slots = {}

    for slot_name, df in slot_candidates.items():

        keep_indices = []

        for idx in range(len(df)):

            candidate_id = (
                f"{slot_name}_{idx}"
            )

            if keep_map.get(
                candidate_id,
                True
            ):
                keep_indices.append(idx)

        filtered_slots[slot_name] = (
            df.iloc[keep_indices]
            .copy()
            .reset_index(drop=True)
        )

    return filtered_slots


filtered_slot_candidates = apply_slot_gate(
    slot_candidates,
    slot_gate_result
)

for slot_name, df in filtered_slot_candidates.items():
    print(
        slot_name,
        ":",
        len(df)
    )

주식 : 11
국 : 11
주찬 : 10
부찬 : 22
김치 : 10


In [221]:
FINAL_POOL_SIZE = {
    "주식": 8,
    "국": 8,
    "주찬": 8,
    "부찬": 12,
    "김치": 8
}

for slot_name in filtered_slot_candidates:

    filtered_slot_candidates[slot_name] = (
        filtered_slot_candidates[slot_name]
        .head(FINAL_POOL_SIZE[slot_name])
        .reset_index(drop=True)
    )

    print(
        slot_name,
        ":",
        len(filtered_slot_candidates[slot_name])
    )

주식 : 8
국 : 8
주찬 : 8
부찬 : 12
김치 : 8


In [222]:
from itertools import product


def build_meal_combinations(slot_candidates):

    combinations = []

    for (
        staple,
        soup,
        main,
        side,
        kimchi
    ) in product(
        slot_candidates["주식"].to_dict("records"),
        slot_candidates["국"].to_dict("records"),
        slot_candidates["주찬"].to_dict("records"),
        slot_candidates["부찬"].to_dict("records"),
        slot_candidates["김치"].to_dict("records")
    ):

        menus = {
            "주식": staple,
            "국": soup,
            "주찬": main,
            "부찬": side,
            "김치": kimchi
        }

        # 같은 메뉴가 두 슬롯에 동시에 들어가는 조합 제거
        menu_codes = [
            menu["menu_fd_Code"]
            for menu in menus.values()
        ]

        if len(menu_codes) != len(set(menu_codes)):
            continue

        combination = {
            "주식": staple["menu_fd_Nm"],
            "국": soup["menu_fd_Nm"],
            "주찬": main["menu_fd_Nm"],
            "부찬": side["menu_fd_Nm"],
            "김치": kimchi["menu_fd_Nm"]
        }

        # 메뉴 코드도 보존
        for slot_name, menu in menus.items():
            combination[
                f"{slot_name}_code"
            ] = menu["menu_fd_Code"]

        # 식단 전체 영양성분 합산
        for nutrient in nutrient_cols:

            values = [
                menu.get(nutrient)
                for menu in menus.values()
            ]

            valid_values = [
                value
                for value in values
                if pd.notna(value)
            ]

            combination[nutrient] = (
                sum(valid_values)
                if valid_values
                else np.nan
            )

        # RAG 의미 유사도 평균
        combination["retrieval_score"] = np.mean([
            menu["retrieval_score"]
            for menu in menus.values()
        ])

        combinations.append(
            combination
        )

    return pd.DataFrame(
        combinations
    )


meal_combinations_df = build_meal_combinations(
    filtered_slot_candidates
)

print(
    "생성된 식단 조합 :",
    len(meal_combinations_df)
)

print(
    "생성된 식단 조합 :",
    len(meal_combinations_df)
)

display(
    meal_combinations_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "단백질",
            "나트륨",
            "retrieval_score"
        ]
    ].head()
)

생성된 식단 조합 : 48128
생성된 식단 조합 : 48128


,주식,국,주찬,부찬,김치,단백질,나트륨,retrieval_score
0,이천영양밥(향토-경기도),계란국,스크램블드에그,호박나물(호박탕쉬),늙은 호박 김치,20.82136,1637.047,0.543916
1,이천영양밥(향토-경기도),계란국,스크램블드에그,호박나물(호박탕쉬),시금치물김치,22.23580,1706.705,0.538595
2,이천영양밥(향토-경기도),계란국,스크램블드에그,호박나물(호박탕쉬),보쌈김치,20.38060,1499.635,0.537649
3,이천영양밥(향토-경기도),계란국,스크램블드에그,호박나물(호박탕쉬),부추김치,20.07078,1551.163,0.537589
4,이천영양밥(향토-경기도),계란국,스크램블드에그,호박나물(호박탕쉬),부추김치,20.35740,1619.135,0.537589


In [223]:
def range_score(
    series,
    lower,
    upper
):
    """
    적정 범위 내부 = 1
    범위 밖 = 거리에 따라 점진적 감점
    """
    score = pd.Series(
        1.0,
        index=series.index
    )

    width = upper - lower

    low_mask = series < lower
    high_mask = series > upper

    score.loc[low_mask] = (
        1
        - (
            lower
            - series.loc[low_mask]
        )
        / width
    )

    score.loc[high_mask] = (
        1
        - (
            series.loc[high_mask]
            - upper
        )
        / width
    )

    return score.clip(
        lower=0,
        upper=1
    )


def rank_meal_combinations(
    meal_df,
    nutrition_high,
    nutrition_low,
    top_k=2000
):

    result = meal_df.copy()

    # --------------------------------
    # 1. 다량영양소 에너지 비율
    # --------------------------------

    result = result[
        result["에너지"] > 0
    ].copy()

    result["carb_energy_ratio"] = (
        result["탄수화물"]
        * 4
        / result["에너지"]
        * 100
    )

    result["protein_energy_ratio"] = (
        result["단백질"]
        * 4
        / result["에너지"]
        * 100
    )

    result["fat_energy_ratio"] = (
        result["지방"]
        * 9
        / result["에너지"]
        * 100
    )

    # 2025 KDRI
    result["carb_balance_score"] = (
        range_score(
            result["carb_energy_ratio"],
            50,
            65
        )
    )

    result["protein_balance_score"] = (
        range_score(
            result["protein_energy_ratio"],
            10,
            20
        )
    )

    result["fat_balance_score"] = (
        range_score(
            result["fat_energy_ratio"],
            15,
            30
        )
    )

    result["macro_balance_score"] = (
        result[
            [
                "carb_balance_score",
                "protein_balance_score",
                "fat_balance_score"
            ]
        ]
        .mean(axis=1)
    )

    # --------------------------------
    # 2. 미량영양소 영양밀도
    #    1000 kcal 기준 상대평가
    # --------------------------------

    beneficial_cols = [
        "총 식이섬유",
        "칼슘",
        "철",
        "마그네슘",
        "칼륨",
        "아연",
        "비타민 A",
        "비타민 D",
        "비타민 C",
        "티아민",
        "리보플라빈",
        "비타민 B12",
        "비타민 K1",
        "오메가3 지방산"
    ]

    beneficial_scores = []

    for col in beneficial_cols:

        if col not in result.columns:
            continue

        density = (
            result[col]
            / result["에너지"]
            * 1000
        )

        if density.notna().any():

            score_col = (
                f"_benefit_{col}"
            )

            result[score_col] = (
                density
                .rank(pct=True)
                .fillna(0.5)
            )

            beneficial_scores.append(
                score_col
            )

    if beneficial_scores:

        result[
            "micronutrient_score"
        ] = (
            result[
                beneficial_scores
            ]
            .mean(axis=1)
        )

    else:

        result[
            "micronutrient_score"
        ] = 0.5

    # --------------------------------
    # 3. 제한 영양소
    # --------------------------------

    limit_cols = [
        "당류",
        "나트륨",
        "총 포화 지방산",
        "총 트랜스 지방산",
        "콜레스테롤"
    ]

    limit_scores = []

    for col in limit_cols:

        if col not in result.columns:
            continue

        density = (
            result[col]
            / result["에너지"]
            * 1000
        )

        if density.notna().any():

            score_col = (
                f"_limit_{col}"
            )

            result[score_col] = (
                1
                - density.rank(
                    pct=True
                )
            ).fillna(0.5)

            limit_scores.append(
                score_col
            )

    if limit_scores:

        result[
            "limit_nutrient_score"
        ] = (
            result[
                limit_scores
            ]
            .mean(axis=1)
        )

    else:

        result[
            "limit_nutrient_score"
        ] = 0.5

    # --------------------------------
    # 4. 에너지 극단값 억제
    # 절대 kcal 기준을 임의로 만들지 않고
    # 현재 후보군의 중앙값 근처를 선호
    # --------------------------------

    energy_pct = (
        result["에너지"]
        .rank(pct=True)
    )

    result[
        "energy_center_score"
    ] = (
        1
        - (
            energy_pct
            - 0.5
        ).abs()
        * 2
    ).clip(
        lower=0,
        upper=1
    )

    # --------------------------------
    # 5. 기본 식단 영양 품질
    # --------------------------------

    result[
        "baseline_nutrition_score"
    ] = (
        result[
            [
                "macro_balance_score",
                "micronutrient_score",
                "limit_nutrient_score",
                "energy_center_score"
            ]
        ]
        .mean(axis=1)
    )

    # --------------------------------
    # 6. 사용자 요청
    # --------------------------------

    preference_scores = []

    for col in nutrition_high:

        if col not in result.columns:
            continue

        score_col = (
            f"_pref_high_{col}"
        )

        if col == "단백질":

            # 단백질을 높게 원하더라도
            # 20%를 넘어 무한 보상하지 않음
            result[score_col] = (
                result[
                    "protein_energy_ratio"
                ]
                .clip(
                    lower=0,
                    upper=20
                )
                / 20
            )

        else:

            density = (
                result[col]
                / result["에너지"]
                * 1000
            )

            result[score_col] = (
                density.rank(
                    pct=True
                )
                .fillna(0.5)
            )

        preference_scores.append(
            score_col
        )

    for col in nutrition_low:

        if col not in result.columns:
            continue

        score_col = (
            f"_pref_low_{col}"
        )

        density = (
            result[col]
            / result["에너지"]
            * 1000
        )

        result[score_col] = (
            1
            - density.rank(
                pct=True
            )
        ).fillna(0.5)

        preference_scores.append(
            score_col
        )

    if preference_scores:

        result[
            "preference_score"
        ] = (
            result[
                preference_scores
            ]
            .mean(axis=1)
        )

        # 기본 영양 균형과 사용자 요구를
        # 동일 비중으로 평가
        result[
            "nutrition_score"
        ] = (
            result[
                [
                    "baseline_nutrition_score",
                    "preference_score"
                ]
            ]
            .mean(axis=1)
        )

    else:

        result[
            "preference_score"
        ] = 0.5

        result[
            "nutrition_score"
        ] = result[
            "baseline_nutrition_score"
        ]

    # --------------------------------
    # 7. 최종 정렬
    # --------------------------------

    return (
        result
        .sort_values(
            [
                "nutrition_score",
                "retrieval_score"
            ],
            ascending=[
                False,
                False
            ]
        )
        .head(top_k)
        .reset_index(drop=True)
    )

In [225]:
ranked_meal_df = rank_meal_combinations(
    meal_combinations_df,
    nutrition_high=conditions["nutrition_high"],
    nutrition_low=conditions["nutrition_low"],
    top_k=2000
)

display(
    ranked_meal_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "단백질",
            "나트륨",
            "nutrition_score",
            "retrieval_score"
        ]
    ].head(10)
)

,주식,국,주찬,부찬,김치,단백질,나트륨,nutrition_score,retrieval_score
0,조감자밥,얼큰청양콩나물국,일본식 계란말이,양하꽃대무침(양애깐무침),시금치물김치,37.000980,1083.281,0.908601,0.530154
1,조감자밥,바지락콩나물국,일본식 계란말이,양하꽃대무침(양애깐무침),시금치물김치,36.654780,1094.311,0.905675,0.528765
2,조감자밥,바지락콩나물국,달걀프리타타,양하꽃대무침(양애깐무침),시금치물김치,33.180315,1043.619,0.897686,0.524030
3,조감자밥,얼큰청양콩나물국,일본식 계란말이,무염 겉절이,시금치물김치,36.076580,1085.231,0.897465,0.530830
4,조감자밥,얼큰청양콩나물국,봄동오리쌈,양하꽃대무침(양애깐무침),시금치물김치,30.122210,1089.134,0.897245,0.524746
5,조감자밥,얼큰청양콩나물국,일본식 계란말이,양하꽃대무침(양애깐무침),늙은 호박 김치,35.586540,1013.623,0.896291,0.535474
6,조감자밥,얼큰청양콩나물국,달걀프리타타,양하꽃대무침(양애깐무침),시금치물김치,33.526515,1032.589,0.895985,0.525420
7,조감자밥,바지락콩나물국,봄동오리쌈,양하꽃대무침(양애깐무침),시금치물김치,29.776010,1100.164,0.895242,0.523357
8,조감자밥,바지락콩나물국,일본식 계란말이,무염 겉절이,시금치물김치,35.730380,1096.261,0.895008,0.529441
9,조감자밥,바지락콩나물국,일본식 계란말이,양하꽃대무침(양애깐무침),늙은 호박 김치,35.240340,1024.653,0.893337,0.534085


In [216]:
# 다양한 식단 선택
MEAL_SLOTS = [
    "주식",
    "국",
    "주찬",
    "부찬",
    "김치"
]


def select_diverse_meals(
    ranked_df,
    n_meals=3,
    max_shared_slots=2
):
    selected = []

    for _, candidate in ranked_df.iterrows():

        # 첫 번째는 최고 점수 식단 그대로 선택
        if not selected:
            selected.append(candidate)
            continue

        is_diverse = True

        for chosen in selected:

            shared_slots = sum(
                candidate[slot] == chosen[slot]
                for slot in MEAL_SLOTS
            )

            # 기존 선택 식단과 너무 많이 겹치면 제외
            if shared_slots > max_shared_slots:
                is_diverse = False
                break

        if is_diverse:
            selected.append(candidate)

        if len(selected) >= n_meals:
            break

    return pd.DataFrame(selected).reset_index(drop=True)

In [230]:
diverse_meal_df = select_diverse_meals(
    ml_ranked_meal_df,
    n_meals=3,
    max_shared_slots=1
)

display(
    diverse_meal_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "단백질",
            "나트륨",
            "nutrition_score",
            "cluster_diversity_score",
            "final_score",
            "retrieval_score"
        ]
    ]
)

,주식,국,주찬,부찬,김치,단백질,나트륨,nutrition_score,cluster_diversity_score,final_score,retrieval_score
0,조감자밥,얼큰청양콩나물국,일본식 계란말이,양하꽃대무침(양애깐무침),시금치물김치,37.000980,1083.281,0.908601,0.8,0.903171,0.530154
1,조감자밥,바지락콩나물국,봄동오리쌈,콩나물잡채,갓김치,37.821110,1339.514,0.889996,0.8,0.885496,0.522901
2,조감자밥,조개탕,달걀프리타타,무염 겉절이,부추김치,34.113515,1188.436,0.877992,0.8,0.874093,0.523325


## 가이드라인 RAG

In [ ]:
# Guideline corpus
guideline_records = [
    {
        "doc_id": "KDRI_PROTEIN_RATIO",
        "source": "보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",
        "topic": "단백질",
        "document": """
대상: 3세 이상
영양소: 단백질
기준: 단백질 에너지적정비율은 총 에너지 섭취량의 10~20%이다.
활용: 식사의 단백질 적정성을 평가할 때 단순히 단백질 g이 높을수록 좋다고 판단하지 않고,
총 에너지 중 단백질이 차지하는 비율을 함께 고려한다.
""".strip()
    },

    {
        "doc_id": "KDRI_PROTEIN_OLDER",
        "source": "보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",
        "topic": "노인 단백질",
        "document": """
대상: 65세 이상 성인
영양소: 단백질
65~74세 남성 권장섭취량은 60 g/일이다.
75세 이상 남성 권장섭취량은 60 g/일이다.
65~74세 여성 권장섭취량은 50 g/일이다.
75세 이상 여성 권장섭취량은 50 g/일이다.
성별이 확인되지 않은 경우 특정 한 값을 개인 기준으로 확정하지 않는다.
""".strip()
    },

    {
        "doc_id": "KDRI_SODIUM_OLDER",
        "source": "보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",
        "topic": "노인 나트륨",
        "document": """
대상: 65세 이상 성인
영양소: 나트륨
65~74세 충분섭취량은 1,300 mg/일이고,
만성질환위험감소섭취량은 1,900 mg/일이다.
75세 이상 충분섭취량은 1,200 mg/일이고,
만성질환위험감소섭취량은 1,800 mg/일이다.
연령이 명확하지 않은 경우 65~74세와 75세 이상 기준의 차이를 고려한다.
""".strip()
    },

    {
        "doc_id": "KSH_HYPERTENSION_SODIUM",
        "source": "대한고혈압학회, 2026 제6판 고혈압 진료지침",
        "topic": "고혈압",
        "document": """
질환: 고혈압
비약물적 생활요법으로 나트륨 섭취 제한이 강하게 권고된다.
체중 조절, 절주, 금연, 규칙적인 신체활동,
건강한 식사와 함께 포괄적인 생활습관 개선을 권고한다.
고혈압이라는 질환명만으로 임의의 음식이나 영양 수치를 생성하지 않고
근거가 있는 영양 기준과 함께 적용한다.
""".strip()
    }
]

guideline_documents_df = pd.DataFrame(
    guideline_records
)

display(guideline_documents_df)

,doc_id,source,topic,document
0,KDRI_PROTEIN_RATIO,"보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",단백질,대상: 3세 이상\n영양소: 단백질\n기준: 단백질 에너지적정비율은 총 에너지 섭취...
1,KDRI_PROTEIN_OLDER,"보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",노인 단백질,대상: 65세 이상 성인\n영양소: 단백질\n65~74세 남성 권장섭취량은 60 g...
2,KDRI_SODIUM_OLDER,"보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",노인 나트륨,"대상: 65세 이상 성인\n영양소: 나트륨\n65~74세 충분섭취량은 1,300 m..."
3,KSH_HYPERTENSION_SODIUM,"대한고혈압학회, 2026 제6판 고혈압 진료지침",고혈압,질환: 고혈압\n비약물적 생활요법으로 나트륨 섭취 제한이 강하게 권고된다.\n체중 ...


In [ ]:
# Guideline 임베딩
guideline_texts = (
    guideline_documents_df[
        "document"
    ]
    .tolist()
)

guideline_output = bge_model.encode(
    guideline_texts,
    batch_size=4,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

guideline_embeddings = np.asarray(
    guideline_output["dense_vecs"],
    dtype="float32"
)

faiss.normalize_L2(
    guideline_embeddings
)

print(
    "Guideline embedding shape :",
    guideline_embeddings.shape
)

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.89it/s]

Guideline embedding shape : (4, 1024)


In [ ]:
# Guideline FAISS
guideline_index = faiss.IndexFlatIP(
    guideline_embeddings.shape[1]
)

guideline_index.add(
    guideline_embeddings
)

print(
    "Guideline documents :",
    guideline_index.ntotal
)

Guideline documents : 4


In [186]:
# Guideline 검색 함수
def retrieve_guidelines(
    conditions,
    top_k=4
):

    query_parts = []

    if conditions.get("target"):
        query_parts.append(
            f"급식 대상: {conditions['target']}"
        )

    if conditions.get("diseases"):
        query_parts.append(
            "건강 상태: "
            + ", ".join(
                conditions["diseases"]
            )
        )

    if conditions.get("nutrition_high"):
        query_parts.append(
            "많이 원하는 영양소: "
            + ", ".join(
                conditions["nutrition_high"]
            )
        )

    if conditions.get("nutrition_low"):
        query_parts.append(
            "적게 원하는 영양소: "
            + ", ".join(
                conditions["nutrition_low"]
            )
        )

    if conditions.get("keywords"):
        query_parts.append(
            "추가 조건: "
            + ", ".join(
                conditions["keywords"]
            )
        )

    query = "\n".join(
        query_parts
    )

    output = bge_model.encode(
        [query],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False
    )

    query_vector = np.asarray(
        output["dense_vecs"],
        dtype="float32"
    )

    faiss.normalize_L2(
        query_vector
    )

    search_k = min(
        top_k,
        guideline_index.ntotal
    )

    scores, indices = (
        guideline_index.search(
            query_vector,
            search_k
        )
    )

    result = (
        guideline_documents_df
        .iloc[indices[0]]
        .copy()
    )

    result["retrieval_score"] = (
        scores[0]
    )

    return result.reset_index(
        drop=True
    )

In [187]:
# 현재 조건으로 검색
retrieved_guidelines_df = (
    retrieve_guidelines(
        conditions,
        top_k=4
    )
)

display(
    retrieved_guidelines_df[
        [
            "doc_id",
            "topic",
            "source",
            "retrieval_score"
        ]
    ]
)

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 12.36it/s]


,doc_id,topic,source,retrieval_score
0,KSH_HYPERTENSION_SODIUM,고혈압,"대한고혈압학회, 2026 제6판 고혈압 진료지침",0.660781
1,KDRI_SODIUM_OLDER,노인 나트륨,"보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",0.649363
2,KDRI_PROTEIN_RATIO,단백질,"보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",0.630244
3,KDRI_PROTEIN_OLDER,노인 단백질,"보건복지부·한국영양학회, 2025 한국인 영양소 섭취기준",0.607741


In [ ]:
# 식단의 단백질 에너지비율 계산
def add_guideline_features(meal_df):
    result = meal_df.copy()

    # 단백질 1g = 4 kcal
    result["protein_energy_ratio"] = np.where(
        (
            result["에너지"].notna()
            & result["단백질"].notna()
            & (result["에너지"] > 0)
        ),
        (
            result["단백질"]
            * 4
            / result["에너지"]
            * 100
        ),
        np.nan
    )

    return result

In [ ]:
meal_combinations_guideline_df = (
    add_guideline_features(
        meal_combinations_df
    )
)

display(
    meal_combinations_guideline_df[
        [
            "주식",
            "국",
            "주찬",
            "부찬",
            "김치",
            "에너지",
            "단백질",
            "protein_energy_ratio",
            "나트륨"
        ]
    ].head()
)

,주식,국,주찬,부찬,김치,에너지,단백질,protein_energy_ratio,나트륨
0,이천영양밥(향토-경기도),영계백숙,스크램블드에그,떡볶이,늙은 호박 김치,1447.470,62.76076,17.343575,1705.617
1,이천영양밥(향토-경기도),영계백숙,스크램블드에그,떡볶이,시금치물김치,1444.480,64.17520,17.771156,1775.275
2,이천영양밥(향토-경기도),영계백숙,스크램블드에그,떡볶이,보쌈김치,1461.170,62.32000,17.060301,1568.205
3,이천영양밥(향토-경기도),영계백숙,스크램블드에그,떡볶이,부추김치,1422.759,62.01018,17.433783,1619.733
4,이천영양밥(향토-경기도),영계백숙,스크램블드에그,떡볶이,부추김치,1427.100,62.29680,17.461089,1687.705


In [195]:
def inspect_meal_components(
    meal_row
):
    records = []

    for slot in [
        "주식",
        "국",
        "주찬",
        "부찬",
        "김치"
    ]:
        code = meal_row[
            f"{slot}_code"
        ]

        menu_row = (
            recommendable_menu_df[
                recommendable_menu_df[
                    "menu_fd_Code"
                ] == code
            ]
            .iloc[0]
        )

        weight_row = (
            menu_df[
                menu_df["fd_Code"]
                == code
            ]
            .iloc[0]
        )

        records.append({
            "슬롯": slot,
            "메뉴": menu_row["menu_fd_Nm"],
            "메뉴중량_g": weight_row["fd_Wgh"],
            "에너지_kcal": menu_row["에너지"],
            "탄수화물_g": menu_row["탄수화물"],
            "단백질_g": menu_row["단백질"],
            "지방_g": menu_row["지방"],
            "나트륨_mg": menu_row["나트륨"]
        })

    return pd.DataFrame(records)

In [ ]:
meal_check_df = inspect_meal_components(
    meal_combinations_guideline_df.iloc[0]
)

display(meal_check_df)

print(
    "\n총 에너지 :",
    meal_check_df["에너지_kcal"].sum()
)

print(
    "총 메뉴 중량 :",
    meal_check_df["메뉴중량_g"].sum()
)

,슬롯,메뉴,메뉴중량_g,에너지_kcal,탄수화물_g,단백질_g,지방_g,나트륨_mg
0,주식,이천영양밥(향토-경기도),99.0,314.86,68.57880,5.55240,0.9232,3.070
1,국,영계백숙,230.5,569.37,58.95895,37.68430,19.3565,258.185
2,주찬,스크램블드에그,56.0,127.34,2.13490,6.38710,9.8710,420.970
3,부찬,떡볶이,450.0,391.00,81.54300,11.07500,2.7880,665.400
4,김치,늙은 호박 김치,91.2,44.90,9.38690,2.06196,0.2824,357.992



총 에너지 : 1447.4700000000003
총 메뉴 중량 : 926.7


In [228]:
from menu_cluster_model import MenuNutritionCluster

# ML 모델 학습
cluster_model = MenuNutritionCluster(
    k_min=3,
    k_max=7,
    random_state=42
)

clustered_menu_df = cluster_model.fit_predict(
    recommendable_menu_df
)

# 메뉴 코드 → 영양 군집
cluster_map = (
    clustered_menu_df
    .set_index("menu_fd_Code")["nutrition_cluster"]
    .to_dict()
)

print("선택된 Cluster 수 :", cluster_model.best_k_)
print(
    "Silhouette Score :",
    round(cluster_model.silhouette_score_, 4)
)

display(cluster_model.cluster_search_)
display(
    cluster_model.cluster_summary(
        clustered_menu_df
    )
)

선택된 Cluster 수 : 7
Silhouette Score : 0.2544


,k,silhouette_score
0,3,0.233515
1,4,0.241410
2,5,0.245770
3,6,0.253289
4,7,0.254390


,nutrition_cluster,menu_count,carb_energy_ratio,protein_energy_ratio,fat_energy_ratio,총 식이섬유_density,나트륨_density,칼슘_density,철_density,마그네슘_density,칼륨_density,아연_density,비타민 A_density,비타민 C_density,비타민 D_density
0,0,670,27.45,45.18,27.83,18.09,6378.53,868.11,18.26,386.71,3184.11,10.60,284.89,54.63,2.42
1,1,838,26.96,18.55,54.30,9.28,3107.59,352.02,6.55,186.70,1442.54,4.80,297.72,36.30,1.99
2,2,1,5.61,85.61,9.47,NaN,NaN,65789.47,59.65,NaN,NaN,NaN,NaN,0.00,NaN
3,3,356,61.33,27.12,21.85,51.25,15681.10,1484.47,25.29,698.91,6275.04,10.01,639.26,171.29,0.75
4,4,995,78.35,10.41,11.85,11.99,1483.84,214.16,4.77,149.53,1416.74,4.15,148.17,81.00,0.35
5,5,78,55.26,32.35,23.23,61.22,22496.90,2200.76,25.30,1101.98,9868.51,13.58,3942.59,363.47,0.73
6,6,4,87.03,13.82,2.87,80.47,4395.29,469.70,179.05,141.41,6660.71,1.37,363.64,6539.15,0.00


In [229]:
ML_SLOTS = [
    "주식",
    "국",
    "주찬",
    "부찬",
    "김치"
]

ml_ranked_meal_df = ranked_meal_df.copy()

# 각 메뉴가 속한 영양 Cluster 연결
cluster_cols = []

for slot in ML_SLOTS:

    cluster_col = f"{slot}_cluster"

    ml_ranked_meal_df[cluster_col] = (
        ml_ranked_meal_df[
            f"{slot}_code"
        ]
        .map(cluster_map)
    )

    cluster_cols.append(
        cluster_col
    )


# 한 식단 안에 서로 다른 영양 Cluster가
# 얼마나 포함되는지 계산
ml_ranked_meal_df[
    "cluster_diversity_score"
] = (
    ml_ranked_meal_df[
        cluster_cols
    ]
    .nunique(axis=1)
    / len(ML_SLOTS)
)


# 기존 영양점수를 핵심으로 유지하고
# ML 영양 다양성을 보조적으로 반영
ml_ranked_meal_df[
    "final_score"
] = (
    0.95
    * ml_ranked_meal_df[
        "nutrition_score"
    ]
    +
    0.05
    * ml_ranked_meal_df[
        "cluster_diversity_score"
    ]
)


ml_ranked_meal_df = (
    ml_ranked_meal_df
    .sort_values(
        [
            "final_score",
            "nutrition_score",
            "retrieval_score"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)